In [1]:
!pip install torch_geometric
!pip install deepchem
!pip install wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.9 MB/s eta 0:00:00


In [1]:
import os
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
os.getcwd()

'/content'

In [3]:
os.chdir('/content/drive/MyDrive/graph neural network-solubility prediction')

In [4]:
# import deepchem as dc
import pandas as pd
import numpy as np
import os
import math
import random
from GCN_model import CustomGCN
from custom_dataset import CustomDataset
from trainer import Trainer
import matplotlib.pyplot as plt
import wandb
import torch
from torch_geometric.loader import DataLoader

Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


In [5]:
wandb.login()

wandb: Currently logged in as: phelchegs (phelchegs-georgia-institute-of-technology). Use `wandb login --relogin` to force relogin


True

In [6]:
os.getcwd()

'/content/drive/MyDrive/graph neural network-solubility prediction'

In [7]:
# os.chdir('..')
os.chdir('data')

In [8]:
df = torch.load('processed_data.pt')

In [9]:
node_dim, edge_dim, inside_dim, dropout = 30, 11, 64, 0.2

In [10]:
model = CustomGCN(node_dim, edge_dim, inside_dim, dropout)
print(model)

CustomGCN(
  (conv1): CustomGCNConv()
  (conv2): CustomGCNConv()
  (conv3): CustomGCNConv()
  (lin): Linear(in_features=16, out_features=4, bias=True)
)


In [11]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

Device:  cuda:0


In [12]:
wandb.init(project="Solubility Prediction")

In [13]:
random.shuffle(df)
train_dataset = df[:int(len(df)*0.85)]
val_dataset = df[int(len(df)*0.85):int(len(df)*0.95)]
test_dataset = df[int(len(df)*0.95):]
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = len(test_dataset), shuffle = True)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
sweep_config = {
    'name': 'GCN_sweep',
    'method': 'bayes',
    'metric': {
        'name': 'AUC',
        'goal': 'maximize'
    },
    'parameters' : {
        'learning_rate': {'min': 0.01, 'max': 0.1},
        'weight_decay': {"min": 1e-5, "max": 1e-2},
        "hidden_dim": {"values": [32, 64, 128, 256] },
        "dropout_p": {"values": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]},
        "epochs": {"values": [10, 15, 20, 25, 30]}
    }

}
sweep_id = wandb.sweep(sweep_config, project="Solubility Prediction")

Create sweep with ID: e9wx89r0
Sweep URL: https://wandb.ai/phelchegs-georgia-institute-of-technology/Solubility%20Prediction/sweeps/e9wx89r0


In [14]:
def hypertune(config = None):
    with wandb.init(config = config):
        config = wandb.config
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        model = CustomGCN(node_dim = 30, edge_dim = 11, inside_dim = config.hidden_dim, dropout = config.dropout_p)
        # model.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr = config.learning_rate, weight_decay = config.weight_decay)
        trainer = Trainer(model, optimizer, train_loader, val_loader, device)
        train_losses, train_scores, valid_losses, valid_scores, valid_acc = trainer.train(epochs = config.epochs)
        params = {}
        for key, value in config.items():
            params[key] = value
        wandb.log(params)
        wandb.log({'AUC': valid_scores[-1]})

In [ ]:
wandb.agent(sweep_id, function=hypertune, count=20)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.
wandb: Agent Starting Run: ylfjk2x2 with config:
wandb: 	dropout_p: 0.3
wandb: 	epochs: 25
wandb: 	hidden_dim: 128
wandb: 	learning_rate: 0.07177946044817297
wandb: 	weight_decay: 0.005587660095392377


Epoch 0 for training:   0%|          | 0/262 [00:00<?, ?batch/s]

Epoch 0 for testing:   0%|          | 0/31 [00:00<?, ?batch/s]

Epoch 1 for training:   0%|          | 0/262 [00:00<?, ?batch/s]

Epoch 1 for testing:   0%|          | 0/31 [00:00<?, ?batch/s]

...

Epoch 29 for training:   0%|          | 0/262 [00:00<?, ?batch/s]

Epoch 29 for testing:   0%|          | 0/31 [00:00<?, ?batch/s]

AUC,▁
dropout_p,▁
epochs,▁
hidden_dim,▁
learning_rate,▁
weight_decay,▁
AUC,0.60861
dropout_p,0.6
epochs,30
hidden_dim,128
learning_rate,0.09757


wandb: Agent Starting Run: 19n2wzx6 with config:
wandb: 	dropout_p: 0.6
wandb: 	epochs: 25
wandb: 	hidden_dim: 64
wandb: 	learning_rate: 0.09914899661994991
wandb: 	weight_decay: 0.005530665423914641


Epoch 0 for training:   0%|          | 0/262 [00:00<?, ?batch/s]